# Predicting Potentially Hazardous Asteroids with NASA's NeoWs API

This project pulls near-Earth object data from NASA's [NeoWs API](https://api.nasa.gov/), converts the JSON into a Pandas DataFrame, and trains a Random Forest classifier to predict `is_potentially_hazardous_asteroid` from basic physical and orbital parameters.

The idea: NASA's hazard assessment likely involves computationally expensive analysis, so a fast classifier could help researchers prioritize which asteroids to inspect first. Because missing a hazardous asteroid is far worse than a false alarm, the model is weighted heavily toward catching the hazardous class.

**Setup**
1. Get a free API key at https://api.nasa.gov/
2. Set it as an environment variable: `export NASA_API_KEY="your_key"` (or paste it into the placeholder below)
3. `pip install numpy pandas requests scikit-learn matplotlib`

**Note:** NASA's rate limits are strict, so the data-fetch step can take around 10 minutes.

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Step 1: Fetch the data

The `browse` endpoint returns 20 asteroids per page. I grab the last 500 pages (~10,000 asteroids). I could request more, but there are daily data limits, and I didn't want to burn through a key before finding a bug.

In [ ]:
# Set the NASA_API_KEY environment variable, or paste your own key in place of the placeholder
NASA_API_KEY = os.environ.get("NASA_API_KEY", "YOUR_NASA_API_KEY_HERE")

url = "https://api.nasa.gov/neo/rest/v1/neo/browse"
params = {"api_key": NASA_API_KEY, "page": 0, "size": 20}

# The first request is only used to find out how many pages exist in total
first_response = requests.get(url, params=params)
first_response.raise_for_status()
total_pages = first_response.json()["page"]["total_pages"]

asteroids = []
for page in range(total_pages - 500, total_pages):
    params["page"] = page
    response = requests.get(url, params=params)
    response.raise_for_status()
    asteroids.extend(response.json()["near_earth_objects"])
    print(f"Retrieved page {page + 1} of {total_pages}", end="\r")

print(f"\nTotal asteroids: {len(asteroids)}")

## Step 2: Inspect the structure

Before converting anything, let's look at how a single record is laid out.

In [ ]:
print(asteroids[0].keys())
print()
print(asteroids[0])

## Step 3: Convert to a DataFrame

Each asteroid has a list of `close_approach_data` entries (past and future flybys). I use the closest approach on record. Several fields can be missing, so everything is pulled with `.get()` and safe defaults.

I picked the features I thought were most relevant: absolute magnitude, estimated diameter (min and max), relative velocity, and miss distance.

In [ ]:
rows = []
for a in asteroids:
    # Use the closest approach this asteroid has ever made
    cad = min(
        a.get("close_approach_data", []),
        key=lambda x: float(x["miss_distance"]["kilometers"]),
        default={},
    )

    rows.append({
        "id": a["id"],
        "name": a["name"],
        "is_hazardous": int(a["is_potentially_hazardous_asteroid"]),
        "absolute_magnitude": a.get("absolute_magnitude_h"),
        "est_diameter_min_km": a.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_min"),
        "est_diameter_max_km": a.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_max"),
        "relative_velocity_km_s": float(cad.get("relative_velocity", {}).get("kilometers_per_second", 0)),
        "miss_distance_km": float(cad.get("miss_distance", {}).get("kilometers", 0)),
    })

df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
df.head()

### Check for missing data

Only a handful of rows have nulls, so I just drop them.

In [ ]:
print(df.isnull().sum())
df = df.dropna()
print()
print(f"Rows after dropping nulls: {len(df)}")
print(df["is_hazardous"].value_counts())

## Step 4: Train / validation split

`name`, `id`, and the target column are excluded from the features.

In [ ]:
features = [
    "absolute_magnitude",
    "est_diameter_min_km",
    "est_diameter_max_km",
    "relative_velocity_km_s",
    "miss_distance_km",
]

x = df[features]
y = df["is_hazardous"]

x_train, x_validate, y_train, y_validate = train_test_split(
    x, y, test_size=0.2, random_state=123456
)

print("Training class counts:")
print(y_train.value_counts())
print("\nValidation class counts:")
print(y_validate.value_counts())

## Step 5: Train the Random Forest

Hazardous asteroids are rare (~3.5% of the data), and correctly flagging them matters much more than correctly flagging harmless ones. So I use `class_weight={0: 1, 1: 50}` to penalize missed hazardous asteroids heavily. The 50x weight came from trial and error.

Sources on class weighting:
- https://www.geeksforgeeks.org/machine-learning/how-does-the-classweight-parameter-in-scikit-learn-work/
- https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html

In [ ]:
asteroid_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=16,
    min_samples_split=25,
    min_samples_leaf=8,
    random_state=314159,
    class_weight={0: 1, 1: 50},
    n_jobs=-1,
)

asteroid_rf.fit(x_train, y_train)
y_predict = asteroid_rf.predict(x_validate)

## Step 6: Evaluate

Overall accuracy is lower than an unweighted model would give (roughly 97-99%), but that's the trade-off: recall on the hazardous class is much better, which is what we care about.

In [ ]:
print(f"Accuracy on training data:   {asteroid_rf.score(x_train, y_train):.2%}")
print(f"Accuracy on validation data: {asteroid_rf.score(x_validate, y_validate):.2%}")
print()
print(classification_report(y_validate, y_predict))

In [ ]:
# Every truly hazardous asteroid in the validation set, with the model's prediction
comparison = pd.DataFrame({"actual": y_validate, "predicted": y_predict})
comparison[comparison["actual"] == 1]

### Confusion matrices

The percentage version is normalized per true class, so the diagonal shows recall for each class. The count version shows the raw numbers.

Sources:
- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html
- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html

In [ ]:
labels = ["Not Hazardous", "Hazardous"]

cm_pct = confusion_matrix(y_validate, y_predict, normalize="true")
disp = ConfusionMatrixDisplay(confusion_matrix=cm_pct, display_labels=labels)
disp.plot(cmap="Blues", values_format=".1%")
disp.ax_.set_xlabel("Predicted")
disp.ax_.set_ylabel("Actual")
plt.title("Confusion Matrix (Percents)")
plt.show()

cm_counts = confusion_matrix(y_validate, y_predict)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_counts, display_labels=labels)
disp.plot(cmap="Greens")
disp.ax_.set_xlabel("Predicted")
disp.ax_.set_ylabel("Actual")
plt.title("Confusion Matrix (Counts)")
plt.show()

## Takeaways

- The model catches the large majority of hazardous asteroids, at the cost of some false positives. For a triage tool, that's the right trade.
- More data (more pages from the API) would likely improve it further.
- Natural next steps: add orbital features from `orbital_data` (eccentricity, inclination, minimum orbit intersection distance), tune the class weight, or try gradient boosting.